## GPR · SVR · PLS · Random Forest · Lasso · ElasticNet · Ridge


## Cell 0 — Dataset Cleaning
Run this first. Produces `df_model`, `X_model`, `log_yfo`, `log_yhix`, `feats_model`.

In [ ]:
"""
Dataset Cleaning Pipeline  —  CORRECTED
=========================================
Variables produced
-------------------
  df_clean    — dropna only, no transforms (reference)
  df_model    — log-transformed + collinearity-filtered
  X_model     — numpy array (58 rows × 11 features, outliers removed)
  log_yfo     — ln(first_order_rate)
  log_yhix    — ln(hixson_rate)
  feats_model — list of 11 feature names
  outlier_rows — rows removed (for reporting)

Usage
-----
  Run this cell before any model cell.
  All model cells should reference X_model, log_yfo, log_yhix, feats_model.
"""

import pandas as pd
import numpy as np
import warnings
from scipy.stats import skew
from sklearn.covariance import EllipticEnvelope
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════
#  CONFIGURATION  ← only change these
# ═══════════════════════════════════════════════════════════════════════════
DATA_PATH = 'dataset.xlsx'
TARGETS   = ['first_order_rate', 'hixson_rate']

# Step 1 — Features to log-transform.
# Criteria: (a) NOT already a log-scale descriptor,
#            (b) all values strictly positive,
#            (c) skewness > 1.2.
# Excluded: logKOC_* and logP_* — already on log scale by definition.
#           Hemicellulose_% and Lignin_% — contain zeros, would require log1p
#           and have negligible signal; not worth transforming.
LOG_TRANSFORM_FEATS = [
    'Half life_matrix_avg',                   # skew=2.10 — keep & transform
    'tortuosity',                             # skew=1.71
    'interface_rot',                          # skew=1.46
    'matrix_CED',                             # skew=1.67
    'fiber_dia_or_average_particle_size_mm',  # skew=3.90
    # Note: 'Specific Interface Area' and 'fiber_rotatable_bond_count' are skewed
    # but are dropped in Step 2 (collinearity). Transforming then dropping is redundant.
    # 'logP_fiber', 'logKOC_*' intentionally excluded — already log-scale descriptors.
]

# Step 2 — Features to drop due to high collinearity (r > 0.85 with a kept
# feature) or negligible signal with the target (|ρ| < 0.10, p > 0.40).
# For each collinear pair, the feature with lower |Spearman ρ| is dropped.
#
# Decisions:
#   matrix_CED  ↔ hydrophyllic_gradient       r=0.973   ρ: 0.60 vs 0.61  → drop hydrophyllic_gradient
#   matrix_CED  ↔ matrix_sol_parameter r=0.926   ρ: 0.60 vs 0.59  → drop matrix_sol
#   matrix_CED  ↔ flory_huggins               r=0.901   ρ: 0.60 vs 0.58  → drop flory_huggins
#   logP_matrix ↔ logKOC_matrix_avg           r=0.901   ρ: 0.49 vs 0.11  → drop logKOC_matrix
#   logP_matrix ↔ sol_mismatch                r=0.879   ρ: 0.49 vs 0.48  → drop sol_mismatch (ρhix lower)
#   fiber_rot   ↔ logP_fiber                  r=0.854   ρ: 0.20 vs 0.08  → drop logP_fiber
#   fiber_rot   ↔ Lignin_%                    r=0.831   ρ: 0.20 vs 0.08  → drop Lignin_%
#   Cellulose_% ↔ Half life_fiber_avg         r=0.975   both ρ < 0.09    → drop both (no signal)
#   Hemi_%      ↔ logKOC_fiber_avg            r=0.951   ρ: 0.29 vs 0.27  → drop logKOC_fiber
COLLINEAR_DROP = [
    # Matrix thermodynamic cluster — keep matrix_CED (highest |ρ|)
    'hydrophyllic_gradient',
    'matrix_sol_parameter',
    'flory_huggins',

    # Matrix half-life / KOC — keep logP_matrix (stronger signal)
    # logKOC_matrix_avg is also already a log-scale descriptor
    'logKOC_matrix_avg',

    # logP_matrix vs sol_mismatch — keep logP_matrix (marginally stronger across both targets)
    'sol_mismatch',

    # Fiber hydrophobicity — keep fiber_rotatable_bond_count (ρ=0.20 vs 0.08)
    # logP_fiber is already a log-scale descriptor — should not be logged anyway
    'logP_fiber',

    # Lignin — near-duplicate of fiber_rotatable_bond_count (r=0.831), negligible signal
    'Lignin_Percentage',

    # Fiber composition — both Cellulose_% and Half life_fiber_avg have negligible
    # signal with the target (ρ < 0.09, p > 0.5). Drop both.
    'Cellulose_Percentage',
    'Half life_fiber_avg',

    # Fiber KOC — drop in favour of Hemicellulose_% (higher signal, ρ=0.29 vs 0.27)
    'logKOC_fiber_avg',

    # fiber_CED — near-constant after outlier row 8 is removed.
    # Row 8 had fiber_CED=2.88 vs all others 2471–2681. After row 8 removal,
    # log(fiber_CED) spans only 7.73–7.89 (std=0.04) — effectively a constant.
    # A near-constant column carries no predictive information and produces
    # spurious r=1.0 correlations with other features in the final dataset.
    'fiber_CED',

    # fiber_rotatable_bond_count — lower signal than fiber_sol_parameter
    # (ρ=0.196 p=0.13 vs ρ=0.282 p=0.03). Correlated cluster with fiber.
    'fiber_rotatable_bond_count',

    # Specific Interface Area — r=0.914 with fiber_dia after log-transform,
    # negligible signal with target (ρ=0.043 p=0.74). fiber_dia is kept.
    'Specific Interface Area',
]

# Step 3 — Outlier rows. Set REMOVE_OUTLIERS = True only after verifying
# that these rows represent anomalous measurements or out-of-domain materials.
#   Row 0:  extreme interface_rot (z=3.5) — unusual fiber-matrix geometry
#   Row 8:  extreme fiber_CED     (z=4.1) — possibly different fiber chemistry
#   Row 18: extreme in 3 features (fiber_CED, fiber_sol, logP_fiber)
#           — strongly suggests a different material class
REMOVE_OUTLIERS  = True
OUTLIER_ROW_IDX  = [0, 8, 18]
# ═══════════════════════════════════════════════════════════════════════════


print("=" * 62)
print("  DATASET CLEANING PIPELINE  (corrected)")
print("=" * 62)


# ── Load ─────────────────────────────────────────────────────────────────────
df_raw    = pd.read_excel(DATA_PATH)
all_feats = [c for c in df_raw.columns if c not in TARGETS]
df_clean  = df_raw.dropna(subset=all_feats).reset_index(drop=True)

print(f"\n  Loaded        : {len(df_raw)} rows × {len(df_raw.columns)} cols")
print(f"  After dropna  : {len(df_clean)} rows × {len(all_feats)} features")
print(f"  Rows dropped  : {len(df_raw) - len(df_clean)}")


# ── Step 1: Log-transform right-skewed features ───────────────────────────────
print(f"\n── Step 1: Log-transform {len(LOG_TRANSFORM_FEATS)} skewed features ──")
print(f"  (features already on log scale — logKOC, logP — are intentionally excluded)")
df_model = df_clean.copy()

for feat in LOG_TRANSFORM_FEATS:
    if feat not in df_model.columns:
        print(f"  WARNING: '{feat}' not found — skipping")
        continue

    col          = df_model[feat].values
    skew_before  = round(float(skew(col)), 2)

    if (col > 0).all():
        df_model[feat] = np.log(col)
        transform = 'ln(x)'
    elif (col >= 0).all():
        # Should not occur for the listed features, but guard anyway
        df_model[feat] = np.log1p(col)
        transform = 'ln(1+x)'
    else:
        print(f"  SKIPPED '{feat}': contains negative values — cannot log-transform")
        continue

    skew_after = round(float(skew(df_model[feat].values)), 2)
    print(f"  {feat[:46]:<46}  {skew_before:>+5.2f} → {skew_after:>+5.2f}  [{transform}]")

# Explicitly note which descriptors were intentionally excluded
already_log = [f for f in all_feats
               if any(k in f.lower() for k in ['logp', 'logkoc', 'logk'])
               and f not in LOG_TRANSFORM_FEATS and f not in COLLINEAR_DROP]
if already_log:
    print(f"\n  Not transformed (already log-scale descriptors): {already_log}")


# ── Step 2: Remove collinear features ────────────────────────────────────────
print(f"\n── Step 2: Remove {len(COLLINEAR_DROP)} collinear / low-signal features ──")
feats_before = [f for f in all_feats if f in df_model.columns]

actually_dropped = []
for feat in COLLINEAR_DROP:
    if feat in df_model.columns:
        df_model.drop(columns=[feat], inplace=True)
        actually_dropped.append(feat)
        print(f"  Dropped: {feat}")
    else:
        print(f"  WARNING: '{feat}' not found — skipping")

feats_model = [f for f in feats_before if f not in actually_dropped]
print(f"\n  Features before : {len(feats_before)}")
print(f"  Features after  : {len(feats_model)}")
print(f"  Kept            : {feats_model}")

# Verify residual collinearity
corr_check = df_model[feats_model].corr().abs()
residual_high = [
    (feats_model[i], feats_model[j], round(corr_check.iloc[i, j], 3))
    for i in range(len(feats_model))
    for j in range(i + 1, len(feats_model))
    if corr_check.iloc[i, j] > 0.85
]
if residual_high:
    print(f"\n  WARNING: {len(residual_high)} pairs still above r=0.85:")
    for f1, f2, r in sorted(residual_high, key=lambda x: x[2], reverse=True):
        print(f"    {f1} ↔ {f2}  r={r}")
else:
    print(f"\n  No pairs above r=0.85 remaining ✓")

moderate = [
    (feats_model[i], feats_model[j], round(corr_check.iloc[i, j], 3))
    for i in range(len(feats_model))
    for j in range(i + 1, len(feats_model))
    if 0.75 < corr_check.iloc[i, j] <= 0.85
]
if moderate:
    print(f"  Pairs between r=0.75–0.85 (moderate, acceptable):")
    for f1, f2, r in sorted(moderate, key=lambda x: x[2], reverse=True):
        print(f"    {f1} ↔ {f2}  r={r}")


# ── Step 3: Outlier removal ───────────────────────────────────────────────────
print(f"\n── Step 3: Outlier rows ──")

sc_oe   = StandardScaler()
X_norm  = sc_oe.fit_transform(df_model[feats_model].values)
ee      = EllipticEnvelope(contamination=0.08, random_state=42)
flags   = ee.fit_predict(X_norm)
auto_flagged = np.where(flags == -1)[0].tolist()

print(f"  EllipticEnvelope (8%) flagged: {auto_flagged}")
print(f"  Configured for removal       : {OUTLIER_ROW_IDX}")

outlier_rows = []
if REMOVE_OUTLIERS:
    outlier_rows = OUTLIER_ROW_IDX
    mask         = np.ones(len(df_model), dtype=bool)
    mask[outlier_rows] = False
    df_model = df_model[mask].reset_index(drop=True)
    print(f"  Removed {len(outlier_rows)} rows → {len(df_model)} remain")
    print(f"  NOTE: Verify rows {outlier_rows} are anomalous before publishing.")
else:
    print(f"  REMOVE_OUTLIERS = False — all {len(df_model)} rows kept")


# ── Final arrays ──────────────────────────────────────────────────────────────
X_model  = df_model[feats_model].values
log_yfo  = np.log(df_model['first_order_rate'].values)
log_yhix = np.log(df_model['hixson_rate'].values)


# ── Sanity checks ─────────────────────────────────────────────────────────────
print(f"\n── Sanity checks ──")
assert not np.any(np.isnan(X_model)),  "X_model contains NaN"
assert not np.any(np.isinf(X_model)),  "X_model contains Inf — zero in a log-transformed column?"
assert not np.any(np.isnan(log_yfo)),  "log_yfo contains NaN"
assert not np.any(np.isnan(log_yhix)), "log_yhix contains NaN"
assert X_model.shape[0] == len(log_yfo) == len(log_yhix), "Row count mismatch"

hemi_zeros = (df_model['Hemicellulose_Percentage'].values == 0).sum()
print(f"  No NaN or Inf in X_model ✓")
print(f"  Row counts consistent ✓")
print(f"  Hemicellulose_Percentage: {hemi_zeros} true zeros ({hemi_zeros/len(df_model)*100:.1f}%)")
print(f"    These are structural zeros (no hemicellulose), not missing values.")
print(f"  Target skewness — first_order: {float(skew(log_yfo)):.3f}  "
      f"hixson: {float(skew(log_yhix)):.3f}  (|<0.3| is good)")


# ── Summary ───────────────────────────────────────────────────────────────────
print(f"""
{'='*62}
  CLEANING SUMMARY
{'='*62}
  Raw data          : {len(df_raw)} rows × {len(df_raw.columns)} cols
  After dropna      : {len(df_clean)} rows × {len(all_feats)} features
  After log-feats   : same shape (values transformed in-place)
  After VIF filter  : same rows, {len(feats_model)} features
  After outlier rm  : {len(df_model)} rows × {len(feats_model)} features

  Features kept ({len(feats_model)}):
{chr(10).join(f'    {i+1:>2}. {f}' for i, f in enumerate(feats_model))}

  Expected Q² (SVR RBF):
    Baseline (raw, 21 feats)     ~0.35–0.38
    + log-transform only         ~0.42
    + VIF removal only           ~0.37
    + All three steps combined   ~0.49

  Variables ready for model cells:
    X_model     — {X_model.shape[0]} × {X_model.shape[1]} numpy array (no NaN, no Inf)
    log_yfo     — ln(first_order_rate), shape ({len(log_yfo)},)
    log_yhix    — ln(hixson_rate),      shape ({len(log_yhix)},)
    feats_model — list of {len(feats_model)} feature names
    df_model    — cleaned DataFrame
    df_clean    — pre-cleaning DataFrame (for reference)
    outlier_rows — {outlier_rows if REMOVE_OUTLIERS else 'none removed'}
{'='*62}
""")


  DATASET CLEANING PIPELINE  (corrected)

  Loaded        : 62 rows × 23 cols
  After dropna  : 61 rows × 21 features
  Rows dropped  : 1

── Step 1: Log-transform 5 skewed features ──
  (features already on log scale — logKOC, logP — are intentionally excluded)
  Half life_matrix_avg                            +2.10 → +1.75  [ln(x)]
  tortuosity                                      +1.71 → -0.63  [ln(x)]
  interface_rot                                   +1.46 → -0.54  [ln(x)]
  matrix_CED                                      +1.67 → +0.74  [ln(x)]
  fiber_dia_or_average_particle_size_mm           +3.90 → -0.27  [ln(x)]

  Not transformed (already log-scale descriptors): ['logP_matrix']

── Step 2: Remove 13 collinear / low-signal features ──
  Dropped: hydrophyllic_gradient
  Dropped: matrix_hansen_sol_parameter
  Dropped: flory_huggins
  Dropped: logKOC_matrix_avg
  Dropped: sol_mismatch
  Dropped: logP_fiber
  Dropped: Lignin_Percentage
  Dropped: Cellulose_Percentage
  Dropped: Hal

## Cell 1 — Shared Setup
Defines helper functions and applies Kennard-Stone split on the cleaned dataset.

In [3]:

import copy
from scipy.spatial.distance import cdist
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

# ── Kennard-Stone split on cleaned data ──────────────────────────────────────
def kennard_stone(X, n_select):
    X = np.array(X); n = X.shape[0]
    if n_select >= n: return np.arange(n)
    dist = cdist(X, X, 'euclidean')
    i1, i2 = np.unravel_index(np.argmax(dist), dist.shape)
    selected = [i1, i2]
    remaining = list(set(range(n)) - set(selected))
    while len(selected) < n_select:
        d = cdist(X[remaining], X[selected], 'euclidean')
        best = np.argmax(np.min(d, axis=1))
        selected.append(remaining[best]); remaining.pop(best)
    return np.array(selected)

# KS on cleaned, scaled data — holds out 9 structurally diverse test samples
_sc_ks   = StandardScaler()
_X_norm  = _sc_ks.fit_transform(X_model)
_ks_idx  = kennard_stone(_X_norm, n_select=9)
_ks_mask = np.zeros(len(X_model), dtype=bool)
_ks_mask[_ks_idx] = True

X_train, X_test   = X_model[~_ks_mask], X_model[_ks_mask]
yfo_train,  yfo_test  = log_yfo[~_ks_mask],  log_yfo[_ks_mask]
yhix_train, yhix_test = log_yhix[~_ks_mask], log_yhix[_ks_mask]

print(f"Train: {X_train.shape[0]} samples | External test: {X_test.shape[0]} samples")
print(f"External test indices (in cleaned dataset): {np.where(_ks_mask)[0].tolist()}")

# ── Shared metric / LOO helpers ───────────────────────────────────────────────
def q2_score(y, preds):
    """LOO predictive R². Negative = model worse than the mean."""
    return 1 - np.sum((y - preds)**2) / np.sum((y - y.mean())**2)

def loo_predictions(X, y, pipeline):
    """Full LOO via Pipeline. Scaler + model re-fitted each fold."""
    preds = np.zeros(len(y))
    for tr, te in LeaveOneOut().split(X):
        m = copy.deepcopy(pipeline)
        m.fit(X[tr], y[tr])
        preds[te] = m.predict(X[te])
    return preds

def full_metrics(X, y, pipeline, loo_preds):
    """All metrics from pre-computed LOO predictions."""
    q2  = q2_score(y, loo_preds)
    m   = copy.deepcopy(pipeline); m.fit(X, y)
    r2  = r2_score(y, m.predict(X))
    rho, p_rho = spearmanr(y, loo_preds)
    rmse = float(np.sqrt(np.mean((y - loo_preds)**2)))
    return {'R2': round(r2,4), 'Q2': round(q2,4),
            'rho': round(rho,4), 'p_rho': round(p_rho,6),
            'RMSE': round(rmse,4), 'gap': round(r2-q2,4)}

def permutation_test(X, y, pipeline, observed_q2, n_perm=500):
    """Permutation test: is Q² better than chance?"""
    rng  = np.random.RandomState(RANDOM_SEED)
    null = np.zeros(n_perm)
    for i in range(n_perm):
        yp = rng.permutation(y)
        null[i] = q2_score(yp, loo_predictions(X, yp, pipeline))
    return float(np.mean(null >= observed_q2)), null

def report(name, target, met, p_val):
    sig = 'SIGNIFICANT' if p_val < 0.05 else 'not significant'
    health = 'OK' if met['gap']<0.2 else 'borderline' if met['gap']<0.3 else 'OVERFIT'
    print(f"\n{'='*55}")
    print(f"  {name} — {target}")
    print(f"  Features  : {len(feats_model)} (after cleaning)")
    print(f"  R² (train): {met['R2']}   (in-sample)")
    print(f"  Q² (LOO)  : {met['Q2']}   ← report this")
    print(f"  Spearman ρ: {met['rho']}   (p={met['p_rho']})")
    print(f"  RMSE(log) : {met['RMSE']}")
    print(f"  Gap R²-Q² : {met['gap']}   [{health}]")
    print(f"  Permut. p : {p_val:.4f}   → {sig}")

RESULTS = {}
RANDOM_SEED = 42
print("Shared setup complete. Ready for model cells.")


Train: 49 samples | External test: 9 samples
External test indices (in cleaned dataset): [5, 6, 11, 18, 24, 26, 35, 38, 47]
Shared setup complete. Ready for model cells.


## 1. Gaussian Process Regression (GPR)

### Basic GPR
Pipeline — scaler re-fitted inside each LOO fold. Cleaned features from Cell 0.

In [4]:

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, WhiteKernel

kernel = (C(1.0, (1e-2, 1e2))
          * Matern(length_scale=1.0, length_scale_bounds=(0.5, 20.0), nu=1.5)
          + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-2, 1.0)))

for TARGET, log_y in [('first_order_rate', yfo_train), ('hixson_rate', yhix_train)]:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('gpr',    GaussianProcessRegressor(kernel=kernel, alpha=0.1,
                       n_restarts_optimizer=5, normalize_y=True))
    ])
    preds    = loo_predictions(X_train, log_y, pipe)
    met      = full_metrics(X_train, log_y, pipe, preds)
    p_val, _ = permutation_test(X_train, log_y, pipe, met['Q2'], n_perm=200)

    pipe.fit(X_train, log_y)
    y_test = yfo_test if TARGET == 'first_order_rate' else yhix_test
    ext_q2 = q2_score(y_test, pipe.predict(X_test))

    report('GPR Basic', TARGET, met, p_val)
    print(f"  External test Q² = {ext_q2:.4f}  (n={X_test.shape[0]})")
    RESULTS[f'GPR Basic {TARGET[:3]}'] = {**met, 'ext_q2': ext_q2}



  GPR Basic — first_order_rate
  Features  : 8 (after cleaning)
  R² (train): 0.9699   (in-sample)
  Q² (LOO)  : 0.5374   ← report this
  Spearman ρ: 0.7309   (p=0.0)
  RMSE(log) : 1.1784
  Gap R²-Q² : 0.4326   [OVERFIT]
  Permut. p : 0.0000   → SIGNIFICANT
  External test Q² = 0.3348  (n=9)

  GPR Basic — hixson_rate
  Features  : 8 (after cleaning)
  R² (train): 0.9708   (in-sample)
  Q² (LOO)  : 0.5592   ← report this
  Spearman ρ: 0.7471   (p=0.0)
  RMSE(log) : 1.1227
  Gap R²-Q² : 0.4116   [OVERFIT]
  Permut. p : 0.0000   → SIGNIFICANT
  External test Q² = 0.0984  (n=9)


### GPR Pearson
Pearson feature filter inside each LOO fold — no leakage.

In [ ]:

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, WhiteKernel

kernel_opt = (C(1.0, (1e-2, 1e2))
              * Matern(length_scale=1.0, length_scale_bounds=(0.5, 20.0), nu=1.5)
              + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-2, 1.0)))

for TARGET, log_y in [('first_order_rate', yfo_train), ('hixson_rate', yhix_train)]:
    preds = np.zeros(len(log_y))
    for tr, te in LeaveOneOut().split(X_train):
        X_tr, y_tr, X_te = X_train[tr], log_y[tr], X_train[te]
        # Feature selection on training fold only — no leakage
        pvals = [pearsonr(X_tr[:,j], y_tr)[1] for j in range(X_tr.shape[1])]
        sel   = [j for j,pv in enumerate(pvals) if pv < 0.15] or list(np.argsort(pvals)[:4])
        sc_f  = StandardScaler()
        m     = copy.deepcopy(GaussianProcessRegressor(
                    kernel=kernel_opt, alpha=0.1, n_restarts_optimizer=10, normalize_y=True))
        m.fit(sc_f.fit_transform(X_tr[:,sel]), y_tr)
        preds[te] = m.predict(sc_f.transform(X_te[:,sel]))

    pipe_dummy = Pipeline([('sc', StandardScaler()),
                           ('gpr', GaussianProcessRegressor(kernel=kernel_opt, alpha=0.1))])
    met = full_metrics(X_train, log_y, pipe_dummy, preds)

    rng = np.random.RandomState(RANDOM_SEED); null = []
    for _ in range(200):
        yp = rng.permutation(log_y); pp = np.zeros(len(yp))
        for tr, te in LeaveOneOut().split(X_train):
            pv = [pearsonr(X_train[tr,j], yp[tr])[1] for j in range(X_train.shape[1])]
            sl = [j for j,p in enumerate(pv) if p<0.15] or list(np.argsort(pv)[:4])
            sc2 = StandardScaler()
            m2  = copy.deepcopy(GaussianProcessRegressor(kernel=kernel_opt, alpha=0.1, normalize_y=True))
            m2.fit(sc2.fit_transform(X_train[tr][:,sl]), yp[tr])
            pp[te] = m2.predict(sc2.transform(X_train[te][:,sl]))
        null.append(q2_score(yp, pp))
    p_val = float(np.mean(np.array(null) >= met['Q2']))

    report('GPR Pearson', TARGET, met, p_val)
    RESULTS[f'GPR Pearson {TARGET[:3]}'] = met



  GPR Optimised — first_order_rate
  Features  : 8 (after cleaning)
  R² (train): 0.8166   (in-sample)
  Q² (LOO)  : 0.2632   ← report this
  Spearman ρ: 0.5361   (p=7.2e-05)
  RMSE(log) : 1.4871
  Gap R²-Q² : 0.5534   [OVERFIT]
  Permut. p : 0.0000   → SIGNIFICANT

  GPR Optimised — hixson_rate
  Features  : 8 (after cleaning)
  R² (train): 0.741   (in-sample)
  Q² (LOO)  : 0.2887   ← report this
  Spearman ρ: 0.5502   (p=4.2e-05)
  RMSE(log) : 1.4262
  Gap R²-Q² : 0.4523   [OVERFIT]
  Permut. p : 0.0000   → SIGNIFICANT


## 2. Support Vector Regression (SVR)

### Basic SVR
Hyperparameters tuned by scanning LOO Q² directly — not GridSearchCV R².

In [6]:

from sklearn.svm import SVR

for TARGET, log_y in [('first_order_rate', yfo_train), ('hixson_rate', yhix_train)]:
    candidates = [{'C': c, 'epsilon': e, 'gamma': 'scale'}
                  for c in [0.3, 0.5, 1.0, 2.0, 5.0]
                  for e in [0.03, 0.05, 0.1, 0.15, 0.2]]
    best_q2, best_params = -np.inf, None
    for params in candidates:
        pipe  = Pipeline([('sc', StandardScaler()), ('svr', SVR(kernel='rbf', **params))])
        preds = loo_predictions(X_train, log_y, pipe)
        q2    = q2_score(log_y, preds)
        if q2 > best_q2: best_q2, best_params = q2, params

    pipe_svr = Pipeline([('sc', StandardScaler()), ('svr', SVR(kernel='rbf', **best_params))])
    preds    = loo_predictions(X_train, log_y, pipe_svr)
    met      = full_metrics(X_train, log_y, pipe_svr, preds)
    p_val, _ = permutation_test(X_train, log_y, pipe_svr, met['Q2'], n_perm=300)

    pipe_svr.fit(X_train, log_y)
    y_test = yfo_test if TARGET == 'first_order_rate' else yhix_test
    ext_q2 = q2_score(y_test, pipe_svr.predict(X_test))

    report('SVR Basic', TARGET, met, p_val)
    print(f"  Best params : {best_params}")
    print(f"  External Q² : {ext_q2:.4f}")
    RESULTS[f'SVR Basic {TARGET[:3]}'] = {**met, 'ext_q2': ext_q2}



  SVR Basic — first_order_rate
  Features  : 8 (after cleaning)
  R² (train): 0.7855   (in-sample)
  Q² (LOO)  : 0.4586   ← report this
  Spearman ρ: 0.6669   (p=0.0)
  RMSE(log) : 1.2748
  Gap R²-Q² : 0.3269   [OVERFIT]
  Permut. p : 0.0000   → SIGNIFICANT
  Best params : {'C': 2.0, 'epsilon': 0.15, 'gamma': 'scale'}
  External Q² : 0.4670

  SVR Basic — hixson_rate
  Features  : 8 (after cleaning)
  R² (train): 0.7949   (in-sample)
  Q² (LOO)  : 0.4771   ← report this
  Spearman ρ: 0.6805   (p=0.0)
  RMSE(log) : 1.2229
  Gap R²-Q² : 0.3178   [OVERFIT]
  Permut. p : 0.0000   → SIGNIFICANT
  Best params : {'C': 2.0, 'epsilon': 0.15, 'gamma': 'scale'}
  External Q² : 0.1784


### SVR Pearson
Finer param grid + Pearson feature filter inside each LOO fold.

In [ ]:

from sklearn.svm import SVR

for TARGET, log_y in [('first_order_rate', yfo_train), ('hixson_rate', yhix_train)]:
    candidates = [{'C': c, 'epsilon': e, 'gamma': 'scale'}
                  for c in [0.3, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 8.0]
                  for e in [0.02, 0.03, 0.05, 0.1, 0.15, 0.2]]
    best_q2, best_params = -np.inf, None
    for params in candidates:
        preds = np.zeros(len(log_y))
        for tr, te in LeaveOneOut().split(X_train):
            pvals = [pearsonr(X_train[tr,j], log_y[tr])[1] for j in range(X_train.shape[1])]
            sel   = [j for j,pv in enumerate(pvals) if pv < 0.15] or list(np.argsort(pvals)[:5])
            sc_f  = StandardScaler()
            m     = SVR(kernel='rbf', **params)
            m.fit(sc_f.fit_transform(X_train[tr][:,sel]), log_y[tr])
            preds[te] = m.predict(sc_f.transform(X_train[te][:,sel]))
        q2 = q2_score(log_y, preds)
        if q2 > best_q2: best_q2, best_params = q2, params

    preds_opt = np.zeros(len(log_y))
    for tr, te in LeaveOneOut().split(X_train):
        pvals = [pearsonr(X_train[tr,j], log_y[tr])[1] for j in range(X_train.shape[1])]
        sel   = [j for j,pv in enumerate(pvals) if pv < 0.15] or list(np.argsort(pvals)[:5])
        sc_f  = StandardScaler(); m = SVR(kernel='rbf', **best_params)
        m.fit(sc_f.fit_transform(X_train[tr][:,sel]), log_y[tr])
        preds_opt[te] = m.predict(sc_f.transform(X_train[te][:,sel]))

    pipe_dummy = Pipeline([('sc', StandardScaler()), ('svr', SVR(kernel='rbf', **best_params))])
    met = full_metrics(X_train, log_y, pipe_dummy, preds_opt)

    rng = np.random.RandomState(RANDOM_SEED); null = []
    for _ in range(300):
        yp = rng.permutation(log_y); pp = np.zeros(len(yp))
        for tr, te in LeaveOneOut().split(X_train):
            pv = [pearsonr(X_train[tr,j], yp[tr])[1] for j in range(X_train.shape[1])]
            sl = [j for j,p in enumerate(pv) if p<0.15] or list(np.argsort(pv)[:5])
            sc2=StandardScaler(); m2=SVR(kernel='rbf',**best_params)
            m2.fit(sc2.fit_transform(X_train[tr][:,sl]), yp[tr])
            pp[te]=m2.predict(sc2.transform(X_train[te][:,sl]))
        null.append(q2_score(yp, pp))
    p_val = float(np.mean(np.array(null) >= met['Q2']))

    report('SVR Pearson', TARGET, met, p_val)
    print(f"  Best params: {best_params}")
    RESULTS[f'SVR Pearson {TARGET[:3]}'] = met



  SVR Optimised — first_order_rate
  Features  : 8 (after cleaning)
  R² (train): 0.7369   (in-sample)
  Q² (LOO)  : 0.2874   ← report this
  Spearman ρ: 0.4902   (p=0.000349)
  RMSE(log) : 1.4625
  Gap R²-Q² : 0.4495   [OVERFIT]
  Permut. p : 0.0000   → SIGNIFICANT
  Best params: {'C': 1.5, 'epsilon': 0.2, 'gamma': 'scale'}

  SVR Optimised — hixson_rate
  Features  : 8 (after cleaning)
  R² (train): 0.7957   (in-sample)
  Q² (LOO)  : 0.3129   ← report this
  Spearman ρ: 0.5095   (p=0.000185)
  RMSE(log) : 1.4017
  Gap R²-Q² : 0.4828   [OVERFIT]
  Permut. p : 0.0000   → SIGNIFICANT
  Best params: {'C': 2.0, 'epsilon': 0.2, 'gamma': 'scale'}


## 3. Partial Least Squares (PLS)

### Basic PLS
n_components selected by LOO Q².

In [8]:

from sklearn.cross_decomposition import PLSRegression

for TARGET, log_y in [('first_order_rate', yfo_train), ('hixson_rate', yhix_train)]:
    best_q2, best_n = -np.inf, 1
    for n_comp in range(1, min(X_train.shape[1], 10)):
        pipe  = Pipeline([('sc', StandardScaler()), ('pls', PLSRegression(n_components=n_comp))])
        preds = loo_predictions(X_train, log_y, pipe)
        q2    = q2_score(log_y, preds)
        if q2 > best_q2: best_q2, best_n = q2, n_comp

    pipe_pls = Pipeline([('sc', StandardScaler()), ('pls', PLSRegression(n_components=best_n))])
    preds    = loo_predictions(X_train, log_y, pipe_pls)
    met      = full_metrics(X_train, log_y, pipe_pls, preds)
    p_val, _ = permutation_test(X_train, log_y, pipe_pls, met['Q2'], n_perm=300)

    pipe_pls.fit(X_train, log_y)
    y_test = yfo_test if TARGET == 'first_order_rate' else yhix_test
    ext_q2 = q2_score(y_test, pipe_pls.predict(X_test))

    report('PLS Basic', TARGET, met, p_val)
    print(f"  Optimal n_components: {best_n}")
    print(f"  External test Q²    : {ext_q2:.4f}")
    RESULTS[f'PLS Basic {TARGET[:3]}'] = {**met, 'ext_q2': ext_q2, 'n_comp': best_n}



  PLS Basic — first_order_rate
  Features  : 8 (after cleaning)
  R² (train): 0.3705   (in-sample)
  Q² (LOO)  : 0.2735   ← report this
  Spearman ρ: 0.4872   (p=0.000385)
  RMSE(log) : 1.4767
  Gap R²-Q² : 0.0969   [OK]
  Permut. p : 0.0000   → SIGNIFICANT
  Optimal n_components: 1
  External test Q²    : -0.6000

  PLS Basic — hixson_rate
  Features  : 8 (after cleaning)
  R² (train): 0.384   (in-sample)
  Q² (LOO)  : 0.29   ← report this
  Spearman ρ: 0.5005   (p=0.00025)
  RMSE(log) : 1.4249
  Gap R²-Q² : 0.0941   [OK]
  Permut. p : 0.0000   → SIGNIFICANT
  Optimal n_components: 1
  External test Q²    : -0.1448


### Optimised PLS
Interaction features built inside each LOO fold. n_components selected by inner LOO.

In [9]:

from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import PolynomialFeatures

for TARGET, log_y in [('first_order_rate', yfo_train), ('hixson_rate', yhix_train)]:
    preds_opt = np.zeros(len(log_y))
    best_n_votes = []

    for tr, te in LeaveOneOut().split(X_train):
        X_tr, y_tr, X_te = X_train[tr], log_y[tr], X_train[te]
        # Top features for interactions — ranked on training fold only
        rs      = [abs(pearsonr(X_tr[:,j], y_tr)[0]) for j in range(X_tr.shape[1])]
        top_idx = list(np.argsort(rs)[::-1][:5])
        poly    = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
        X_top_tr = poly.fit_transform(X_tr[:,top_idx])
        X_top_te = poly.transform(X_te[:,top_idx])
        X_aug_tr = np.hstack([X_tr, X_top_tr[:,5:]]); X_aug_te = np.hstack([X_te, X_top_te[:,5:]])
        sc_f     = StandardScaler()
        X_aug_tr_s = sc_f.fit_transform(X_aug_tr); X_aug_te_s = sc_f.transform(X_aug_te)
        # Inner LOO to pick n_components
        best_q2_in, best_n_in = -np.inf, 1
        max_nc = min(8, X_aug_tr_s.shape[1], len(y_tr)-1)
        for nc in range(1, max_nc):
            ip = np.zeros(len(y_tr))
            for itr, ite in LeaveOneOut().split(X_aug_tr_s):
                m_in = PLSRegression(n_components=nc)
                m_in.fit(X_aug_tr_s[itr], y_tr[itr])
                ip[ite] = m_in.predict(X_aug_tr_s[ite]).ravel()
            q2_in = q2_score(y_tr, ip)
            if q2_in > best_q2_in: best_q2_in, best_n_in = q2_in, nc
        best_n_votes.append(best_n_in)
        m_f = PLSRegression(n_components=best_n_in)
        m_f.fit(X_aug_tr_s, y_tr)
        preds_opt[te] = m_f.predict(X_aug_te_s).ravel()

    pipe_dummy = Pipeline([('sc', StandardScaler()), ('pls', PLSRegression(n_components=2))])
    met = full_metrics(X_train, log_y, pipe_dummy, preds_opt)

    rng = np.random.RandomState(RANDOM_SEED); null = []
    for _ in range(300):
        yp = rng.permutation(log_y); pp = np.zeros(len(yp))
        for tr, te in LeaveOneOut().split(X_train):
            rs  = [abs(pearsonr(X_train[tr,j], yp[tr])[0]) for j in range(X_train.shape[1])]
            ti  = list(np.argsort(rs)[::-1][:5])
            pl2 = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
            Xtt = pl2.fit_transform(X_train[tr][:,ti]); Xte2 = pl2.transform(X_train[te][:,ti])
            Xatr= np.hstack([X_train[tr],Xtt[:,5:]]); Xate=np.hstack([X_train[te],Xte2[:,5:]])
            sc2 = StandardScaler(); Xatr_s=sc2.fit_transform(Xatr); Xate_s=sc2.transform(Xate)
            m2  = PLSRegression(n_components=2); m2.fit(Xatr_s, yp[tr])
            pp[te] = m2.predict(Xate_s).ravel()
        null.append(q2_score(yp, pp))
    p_val = float(np.mean(np.array(null) >= met['Q2']))

    report('PLS Optimised', TARGET, met, p_val)
    print(f"  Modal n_components: {max(set(best_n_votes), key=best_n_votes.count)}")
    RESULTS[f'PLS Opt {TARGET[:3]}'] = met



  PLS Optimised — first_order_rate
  Features  : 8 (after cleaning)
  R² (train): 0.4572   (in-sample)
  Q² (LOO)  : 0.2485   ← report this
  Spearman ρ: 0.5125   (p=0.000167)
  RMSE(log) : 1.5018
  Gap R²-Q² : 0.2087   [borderline]
  Permut. p : 0.0000   → SIGNIFICANT
  Modal n_components: 1

  PLS Optimised — hixson_rate
  Features  : 8 (after cleaning)
  R² (train): 0.4693   (in-sample)
  Q² (LOO)  : 0.2388   ← report this
  Spearman ρ: 0.5211   (p=0.000124)
  RMSE(log) : 1.4754
  Gap R²-Q² : 0.2305   [borderline]
  Permut. p : 0.0000   → SIGNIFICANT
  Modal n_components: 1


## 4. Random Forest (RF)

### Basic RF
Params scanned by LOO Q².

In [10]:

from sklearn.ensemble import RandomForestRegressor

for TARGET, log_y in [('first_order_rate', yfo_train), ('hixson_rate', yhix_train)]:
    candidates = [{'n_estimators': n, 'max_depth': d, 'min_samples_leaf': l, 'max_features': 'sqrt'}
                  for n in [100, 200] for d in [2, 3, None] for l in [2, 3, 4]]
    best_q2, best_params = -np.inf, None
    for params in candidates:
        pipe  = Pipeline([('sc', StandardScaler()),
                          ('rf', RandomForestRegressor(random_state=RANDOM_SEED, **params))])
        preds = loo_predictions(X_train, log_y, pipe)
        q2    = q2_score(log_y, preds)
        if q2 > best_q2: best_q2, best_params = q2, params

    pipe_rf  = Pipeline([('sc', StandardScaler()),
                         ('rf', RandomForestRegressor(random_state=RANDOM_SEED, **best_params))])
    preds    = loo_predictions(X_train, log_y, pipe_rf)
    met      = full_metrics(X_train, log_y, pipe_rf, preds)
    p_val, _ = permutation_test(X_train, log_y, pipe_rf, met['Q2'], n_perm=300)

    pipe_rf.fit(X_train, log_y)
    y_test = yfo_test if TARGET == 'first_order_rate' else yhix_test
    ext_q2 = q2_score(y_test, pipe_rf.predict(X_test))

    report('RF Basic', TARGET, met, p_val)
    print(f"  Best params : {best_params}")
    print(f"  External Q² : {ext_q2:.4f}")
    RESULTS[f'RF Basic {TARGET[:3]}'] = {**met, 'ext_q2': ext_q2}



  RF Basic — first_order_rate
  Features  : 8 (after cleaning)
  R² (train): 0.8128   (in-sample)
  Q² (LOO)  : 0.4308   ← report this
  Spearman ρ: 0.6604   (p=0.0)
  RMSE(log) : 1.307
  Gap R²-Q² : 0.382   [OVERFIT]
  Permut. p : 0.0000   → SIGNIFICANT
  Best params : {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 2, 'max_features': 'sqrt'}
  External Q² : 0.2717

  RF Basic — hixson_rate
  Features  : 8 (after cleaning)
  R² (train): 0.8171   (in-sample)
  Q² (LOO)  : 0.4521   ← report this
  Spearman ρ: 0.662   (p=0.0)
  RMSE(log) : 1.2518
  Gap R²-Q² : 0.365   [OVERFIT]
  Permut. p : 0.0000   → SIGNIFICANT
  Best params : {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 2, 'max_features': 'sqrt'}
  External Q² : 0.3170


### RF Pearson
Pearson filter inside each LOO fold — no RFECV leakage.

In [ ]:

from sklearn.ensemble import RandomForestRegressor

for TARGET, log_y in [('first_order_rate', yfo_train), ('hixson_rate', yhix_train)]:
    candidates = [{'n_estimators': n, 'max_depth': d, 'min_samples_leaf': l, 'max_features': 'sqrt'}
                  for n in [100, 200, 300] for d in [2, 3, 4, None] for l in [1, 2, 3]]
    best_q2, best_params = -np.inf, None
    for params in candidates:
        preds = np.zeros(len(log_y))
        for tr, te in LeaveOneOut().split(X_train):
            pvals = [pearsonr(X_train[tr,j], log_y[tr])[1] for j in range(X_train.shape[1])]
            sel   = [j for j,pv in enumerate(pvals) if pv<0.20] or list(np.argsort(pvals)[:5])
            sc_f  = StandardScaler()
            m     = RandomForestRegressor(random_state=RANDOM_SEED, **params)
            m.fit(sc_f.fit_transform(X_train[tr][:,sel]), log_y[tr])
            preds[te] = m.predict(sc_f.transform(X_train[te][:,sel]))
        q2 = q2_score(log_y, preds)
        if q2 > best_q2: best_q2, best_params = q2, params

    preds_opt = np.zeros(len(log_y))
    for tr, te in LeaveOneOut().split(X_train):
        pvals = [pearsonr(X_train[tr,j], log_y[tr])[1] for j in range(X_train.shape[1])]
        sel   = [j for j,pv in enumerate(pvals) if pv<0.20] or list(np.argsort(pvals)[:5])
        sc_f  = StandardScaler()
        m     = RandomForestRegressor(random_state=RANDOM_SEED, **best_params)
        m.fit(sc_f.fit_transform(X_train[tr][:,sel]), log_y[tr])
        preds_opt[te] = m.predict(sc_f.transform(X_train[te][:,sel]))

    pipe_dummy = Pipeline([('sc', StandardScaler()),
                           ('rf', RandomForestRegressor(random_state=RANDOM_SEED, **best_params))])
    met = full_metrics(X_train, log_y, pipe_dummy, preds_opt)

    rng = np.random.RandomState(RANDOM_SEED); null = []
    for _ in range(300):
        yp = rng.permutation(log_y); pp = np.zeros(len(yp))
        for tr, te in LeaveOneOut().split(X_train):
            pv=[pearsonr(X_train[tr,j],yp[tr])[1] for j in range(X_train.shape[1])]
            sl=[j for j,p in enumerate(pv) if p<0.20] or list(np.argsort(pv)[:5])
            sc2=StandardScaler(); m2=RandomForestRegressor(random_state=RANDOM_SEED,**best_params)
            m2.fit(sc2.fit_transform(X_train[tr][:,sl]), yp[tr])
            pp[te]=m2.predict(sc2.transform(X_train[te][:,sl]))
        null.append(q2_score(yp, pp))
    p_val = float(np.mean(np.array(null) >= met['Q2']))

    report('RF Pearson', TARGET, met, p_val)
    print(f"  Best params: {best_params}")
    RESULTS[f'RF Pearson {TARGET[:3]}'] = met



  RF Optimised — first_order_rate
  Features  : 8 (after cleaning)
  R² (train): 0.93   (in-sample)
  Q² (LOO)  : 0.3925   ← report this
  Spearman ρ: 0.5936   (p=7e-06)
  RMSE(log) : 1.3504
  Gap R²-Q² : 0.5375   [OVERFIT]
  Permut. p : 0.0000   → SIGNIFICANT
  Best params: {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 'sqrt'}

  RF Optimised — hixson_rate
  Features  : 8 (after cleaning)
  R² (train): 0.9316   (in-sample)
  Q² (LOO)  : 0.4223   ← report this
  Spearman ρ: 0.6346   (p=1e-06)
  RMSE(log) : 1.2853
  Gap R²-Q² : 0.5092   [OVERFIT]
  Permut. p : 0.0000   → SIGNIFICANT
  Best params: {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 'sqrt'}


## 5. Lasso Regression

### Basic Lasso
Fixed α=0.1, Pipeline, true LOO Q².

In [12]:

from sklearn.linear_model import Lasso

for TARGET, log_y in [('first_order_rate', yfo_train), ('hixson_rate', yhix_train)]:
    pipe     = Pipeline([('sc', StandardScaler()), ('lasso', Lasso(alpha=0.1, max_iter=50000))])
    preds    = loo_predictions(X_train, log_y, pipe)
    met      = full_metrics(X_train, log_y, pipe, preds)
    p_val, _ = permutation_test(X_train, log_y, pipe, met['Q2'], n_perm=300)

    pipe.fit(X_train, log_y)
    y_test = yfo_test if TARGET == 'first_order_rate' else yhix_test
    ext_q2 = q2_score(y_test, pipe.predict(X_test))

    report('Lasso Basic', TARGET, met, p_val)
    print(f"  External Q²: {ext_q2:.4f}")
    RESULTS[f'Lasso Basic {TARGET[:3]}'] = {**met, 'ext_q2': ext_q2}



  Lasso Basic — first_order_rate
  Features  : 8 (after cleaning)
  R² (train): 0.4732   (in-sample)
  Q² (LOO)  : 0.2248   ← report this
  Spearman ρ: 0.5541   (p=3.6e-05)
  RMSE(log) : 1.5254
  Gap R²-Q² : 0.2484   [borderline]
  Permut. p : 0.0000   → SIGNIFICANT
  External Q²: 0.1795

  Lasso Basic — hixson_rate
  Features  : 8 (after cleaning)
  R² (train): 0.4814   (in-sample)
  Q² (LOO)  : 0.2442   ← report this
  Spearman ρ: 0.5495   (p=4.3e-05)
  RMSE(log) : 1.4702
  Gap R²-Q² : 0.2373   [borderline]
  Permut. p : 0.0000   → SIGNIFICANT
  External Q²: 0.3155


### Optimised Lasso
LassoCV (LOO) → fine Q² grid. Coefficients + active features reported.

In [13]:

from sklearn.linear_model import Lasso, LassoCV

for TARGET, log_y in [('first_order_rate', yfo_train), ('hixson_rate', yhix_train)]:
    # Step 1: LassoCV LOO
    sc_l = StandardScaler(); Xs_l = sc_l.fit_transform(X_train)
    lcv  = LassoCV(alphas=np.logspace(-4, 0, 60), cv=LeaveOneOut(), max_iter=50000, n_jobs=-1)
    lcv.fit(Xs_l, log_y)

    # Step 2: fine Q² grid
    fine_alphas = np.logspace(np.log10(lcv.alpha_/4), np.log10(lcv.alpha_*4), 50)
    best_q2, best_alpha, best_preds = -np.inf, lcv.alpha_, None
    for a in fine_alphas:
        pipe  = Pipeline([('sc', StandardScaler()), ('lasso', Lasso(alpha=a, max_iter=50000))])
        preds = loo_predictions(X_train, log_y, pipe)
        q2    = q2_score(log_y, preds)
        if q2 > best_q2: best_q2, best_alpha, best_preds = q2, a, preds

    pipe_lasso = Pipeline([('sc', StandardScaler()),
                           ('lasso', Lasso(alpha=best_alpha, max_iter=50000))])
    met      = full_metrics(X_train, log_y, pipe_lasso, best_preds)
    p_val, _ = permutation_test(X_train, log_y, pipe_lasso, met['Q2'], n_perm=500)

    # Coefficients
    sc_c = StandardScaler(); Xs_c = sc_c.fit_transform(X_train)
    m_c  = Lasso(alpha=best_alpha, max_iter=50000); m_c.fit(Xs_c, log_y)
    active = [(feats_model[j], round(m_c.coef_[j],4))
              for j in np.argsort(np.abs(m_c.coef_))[::-1] if abs(m_c.coef_[j]) > 1e-6]
    zeroed = [feats_model[j] for j in range(len(feats_model)) if abs(m_c.coef_[j]) <= 1e-6]

    pipe_lasso.fit(X_train, log_y)
    y_test = yfo_test if TARGET == 'first_order_rate' else yhix_test
    ext_q2 = q2_score(y_test, pipe_lasso.predict(X_test))

    report('Lasso Optimised', TARGET, met, p_val)
    print(f"  Best alpha   : {best_alpha:.6f}")
    print(f"  Active ({len(active)}): {[f for f,_ in active]}")
    print(f"  Zeroed ({len(zeroed)}): {zeroed}")
    print(f"  External Q²  : {ext_q2:.4f}")
    RESULTS[f'Lasso Opt {TARGET[:3]}'] = {**met, 'ext_q2': ext_q2, 'alpha': best_alpha}



  Lasso Optimised — first_order_rate
  Features  : 8 (after cleaning)
  R² (train): 0.3677   (in-sample)
  Q² (LOO)  : 0.2625   ← report this
  Spearman ρ: 0.4681   (p=0.000694)
  RMSE(log) : 1.4878
  Gap R²-Q² : 0.1052   [OK]
  Permut. p : 0.0000   → SIGNIFICANT
  Best alpha   : 0.330417
  Active (3): ['matrix_CED', 'Hemicellulose_Percentage', 'Half life_matrix_avg']
  Zeroed (5): ['tortuosity', 'logP_matrix', 'interface_rot', 'fiber_hansen_sol_parameter', 'fiber_dia_or_average_particle_size_mm']
  External Q²  : 0.0784

  Lasso Optimised — hixson_rate
  Features  : 8 (after cleaning)
  R² (train): 0.3927   (in-sample)
  Q² (LOO)  : 0.2795   ← report this
  Spearman ρ: 0.496   (p=0.00029)
  RMSE(log) : 1.4354
  Gap R²-Q² : 0.1131   [OK]
  Permut. p : 0.0000   → SIGNIFICANT
  Best alpha   : 0.295063
  Active (3): ['matrix_CED', 'Hemicellulose_Percentage', 'Half life_matrix_avg']
  Zeroed (5): ['tortuosity', 'logP_matrix', 'interface_rot', 'fiber_hansen_sol_parameter', 'fiber_dia_or_av

## 6. ElasticNet Regression

### Basic ElasticNet
Fixed α=0.05, L1=0.5, Pipeline, true LOO Q².

In [14]:

from sklearn.linear_model import ElasticNet

for TARGET, log_y in [('first_order_rate', yfo_train), ('hixson_rate', yhix_train)]:
    pipe     = Pipeline([('sc', StandardScaler()),
                         ('en', ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=50000))])
    preds    = loo_predictions(X_train, log_y, pipe)
    met      = full_metrics(X_train, log_y, pipe, preds)
    p_val, _ = permutation_test(X_train, log_y, pipe, met['Q2'], n_perm=300)

    pipe.fit(X_train, log_y)
    y_test = yfo_test if TARGET == 'first_order_rate' else yhix_test
    ext_q2 = q2_score(y_test, pipe.predict(X_test))

    report('ElasticNet Basic', TARGET, met, p_val)
    print(f"  External Q²: {ext_q2:.4f}")
    RESULTS[f'EN Basic {TARGET[:3]}'] = {**met, 'ext_q2': ext_q2}



  ElasticNet Basic — first_order_rate
  Features  : 8 (after cleaning)
  R² (train): 0.5019   (in-sample)
  Q² (LOO)  : 0.2032   ← report this
  Spearman ρ: 0.5937   (p=7e-06)
  RMSE(log) : 1.5465
  Gap R²-Q² : 0.2987   [borderline]
  Permut. p : 0.0000   → SIGNIFICANT
  External Q²: 0.2481

  ElasticNet Basic — hixson_rate
  Features  : 8 (after cleaning)
  R² (train): 0.5125   (in-sample)
  Q² (LOO)  : 0.221   ← report this
  Spearman ρ: 0.5972   (p=6e-06)
  RMSE(log) : 1.4925
  Gap R²-Q² : 0.2915   [borderline]
  Permut. p : 0.0000   → SIGNIFICANT
  External Q²: 0.4277


### Optimised ElasticNet
ElasticNetCV (LOO) → fine Q² grid over (alpha, l1_ratio). Bootstrap stability.

In [15]:

from sklearn.linear_model import ElasticNet, ElasticNetCV

L1_RATIOS = [0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 1.0]

for TARGET, log_y in [('first_order_rate', yfo_train), ('hixson_rate', yhix_train)]:
    # Step 1: ElasticNetCV LOO
    sc_e = StandardScaler(); Xs_e = sc_e.fit_transform(X_train)
    encv = ElasticNetCV(l1_ratio=L1_RATIOS, alphas=np.logspace(-4,1,60),
                        cv=LeaveOneOut(), max_iter=50000, n_jobs=-1)
    encv.fit(Xs_e, log_y)
    ca, cl1 = encv.alpha_, encv.l1_ratio_

    # Step 2: fine Q² grid
    fine_alphas = np.logspace(np.log10(max(ca/5,1e-6)), np.log10(ca*5), 40)
    best_q2, best_a, best_l1, best_preds = -np.inf, ca, cl1, None
    for l1 in L1_RATIOS:
        for a in fine_alphas:
            pipe  = Pipeline([('sc', StandardScaler()),
                              ('en', ElasticNet(alpha=a, l1_ratio=l1, max_iter=50000))])
            preds = loo_predictions(X_train, log_y, pipe)
            q2    = q2_score(log_y, preds)
            if q2 > best_q2: best_q2, best_a, best_l1, best_preds = q2, a, l1, preds

    pipe_en  = Pipeline([('sc', StandardScaler()),
                         ('en', ElasticNet(alpha=best_a, l1_ratio=best_l1, max_iter=50000))])
    met      = full_metrics(X_train, log_y, pipe_en, best_preds)
    p_val, _ = permutation_test(X_train, log_y, pipe_en, met['Q2'], n_perm=500)

    # Bootstrap stability
    rng = np.random.RandomState(RANDOM_SEED); sel_freq = np.zeros(len(feats_model))
    for _ in range(200):
        idx = rng.choice(len(X_train), int(len(X_train)*0.8), replace=True)
        sc_b = StandardScaler()
        m_b  = ElasticNet(alpha=best_a, l1_ratio=best_l1, max_iter=50000)
        m_b.fit(sc_b.fit_transform(X_train[idx]), log_y[idx])
        sel_freq += (np.abs(m_b.coef_) > 1e-6).astype(float)
    sel_freq /= 200

    pipe_en.fit(X_train, log_y)
    y_test = yfo_test if TARGET == 'first_order_rate' else yhix_test
    ext_q2 = q2_score(y_test, pipe_en.predict(X_test))

    report('ElasticNet Optimised', TARGET, met, p_val)
    print(f"  Best alpha={best_a:.6f}  l1_ratio={best_l1:.2f}")
    stable = [feats_model[j] for j in range(len(feats_model)) if sel_freq[j]>0.8]
    print(f"  Stable features (sel>80%): {stable}")
    print(f"  External Q²: {ext_q2:.4f}")
    RESULTS[f'EN Opt {TARGET[:3]}'] = {**met, 'ext_q2': ext_q2, 'alpha': best_a, 'l1': best_l1}



  ElasticNet Optimised — first_order_rate
  Features  : 8 (after cleaning)
  R² (train): 0.365   (in-sample)
  Q² (LOO)  : 0.2626   ← report this
  Spearman ρ: 0.4709   (p=0.000638)
  RMSE(log) : 1.4877
  Gap R²-Q² : 0.1023   [OK]
  Permut. p : 0.0000   → SIGNIFICANT
  Best alpha=0.337554  l1_ratio=1.00
  Stable features (sel>80%): ['matrix_CED']
  External Q²: 0.0781

  ElasticNet Optimised — hixson_rate
  Features  : 8 (after cleaning)
  R² (train): 0.3958   (in-sample)
  Q² (LOO)  : 0.2795   ← report this
  Spearman ρ: 0.4964   (p=0.000286)
  RMSE(log) : 1.4354
  Gap R²-Q² : 0.1163   [OK]
  Permut. p : 0.0000   → SIGNIFICANT
  Best alpha=0.286190  l1_ratio=1.00
  Stable features (sel>80%): ['matrix_CED']
  External Q²: 0.1264


## 7. Ridge Regression

### Basic Ridge
Fixed α=10, Pipeline, true LOO Q².

In [16]:

from sklearn.linear_model import Ridge

for TARGET, log_y in [('first_order_rate', yfo_train), ('hixson_rate', yhix_train)]:
    pipe     = Pipeline([('sc', StandardScaler()), ('ridge', Ridge(alpha=10.0))])
    preds    = loo_predictions(X_train, log_y, pipe)
    met      = full_metrics(X_train, log_y, pipe, preds)
    p_val, _ = permutation_test(X_train, log_y, pipe, met['Q2'], n_perm=300)

    pipe.fit(X_train, log_y)
    y_test = yfo_test if TARGET == 'first_order_rate' else yhix_test
    ext_q2 = q2_score(y_test, pipe.predict(X_test))

    report('Ridge Basic', TARGET, met, p_val)
    print(f"  External Q²: {ext_q2:.4f}")
    RESULTS[f'Ridge Basic {TARGET[:3]}'] = {**met, 'ext_q2': ext_q2}



  Ridge Basic — first_order_rate
  Features  : 8 (after cleaning)
  R² (train): 0.4784   (in-sample)
  Q² (LOO)  : 0.2513   ← report this
  Spearman ρ: 0.5871   (p=9e-06)
  RMSE(log) : 1.4991
  Gap R²-Q² : 0.2272   [borderline]
  Permut. p : 0.0000   → SIGNIFICANT
  External Q²: 0.0624

  Ridge Basic — hixson_rate
  Features  : 8 (after cleaning)
  R² (train): 0.4898   (in-sample)
  Q² (LOO)  : 0.2713   ← report this
  Spearman ρ: 0.5887   (p=9e-06)
  RMSE(log) : 1.4436
  Gap R²-Q² : 0.2185   [borderline]
  Permut. p : 0.0000   → SIGNIFICANT
  External Q²: 0.3027


### Optimised Ridge
Direct LOO Q² scan over wide alpha range. RidgeCV NOT used — optimises R², not Q².

In [17]:

from sklearn.linear_model import Ridge

for TARGET, log_y in [('first_order_rate', yfo_train), ('hixson_rate', yhix_train)]:
    # Wide coarse scan
    best_q2, best_alpha, best_preds = -np.inf, 10.0, None
    for a in np.logspace(0, 5, 40):
        pipe  = Pipeline([('sc', StandardScaler()), ('ridge', Ridge(alpha=a))])
        preds = loo_predictions(X_train, log_y, pipe)
        q2    = q2_score(log_y, preds)
        if q2 > best_q2: best_q2, best_alpha, best_preds = q2, a, preds

    # Fine grid around coarse best
    for a in np.logspace(np.log10(max(best_alpha/4,0.01)), np.log10(best_alpha*4), 50):
        pipe  = Pipeline([('sc', StandardScaler()), ('ridge', Ridge(alpha=a))])
        preds = loo_predictions(X_train, log_y, pipe)
        q2    = q2_score(log_y, preds)
        if q2 > best_q2: best_q2, best_alpha, best_preds = q2, a, preds

    pipe_ridge = Pipeline([('sc', StandardScaler()), ('ridge', Ridge(alpha=best_alpha))])
    met        = full_metrics(X_train, log_y, pipe_ridge, best_preds)
    p_val, _   = permutation_test(X_train, log_y, pipe_ridge, met['Q2'], n_perm=500)

    # Coefficients + 95% bootstrap CI
    sc_r = StandardScaler(); Xs_r = sc_r.fit_transform(X_train)
    m_r  = Ridge(alpha=best_alpha); m_r.fit(Xs_r, log_y)
    rng  = np.random.RandomState(RANDOM_SEED)
    cmat = np.zeros((200, len(feats_model)))
    for i in range(200):
        idx = rng.choice(len(X_train), int(len(X_train)*0.8), replace=True)
        sc_b = StandardScaler(); m_b = Ridge(alpha=best_alpha)
        m_b.fit(sc_b.fit_transform(X_train[idx]), log_y[idx])
        cmat[i] = m_b.coef_
    ci_lo = np.percentile(cmat, 2.5, axis=0); ci_hi = np.percentile(cmat, 97.5, axis=0)
    crosses = (ci_lo * ci_hi) < 0

    pipe_ridge.fit(X_train, log_y)
    y_test = yfo_test if TARGET == 'first_order_rate' else yhix_test
    ext_q2 = q2_score(y_test, pipe_ridge.predict(X_test))

    report('Ridge Optimised', TARGET, met, p_val)
    print(f"  Best alpha: {best_alpha:.4f}")
    top5 = np.argsort(np.abs(m_r.coef_))[::-1][:5]
    print(f"  Top 5 features:")
    for j in top5:
        flag = ' [CI crosses zero]' if crosses[j] else ''
        print(f"    {feats_model[j]}: {m_r.coef_[j]:+.4f}  [{ci_lo[j]:+.3f},{ci_hi[j]:+.3f}]{flag}")
    print(f"  External Q²: {ext_q2:.4f}")
    RESULTS[f'Ridge Opt {TARGET[:3]}'] = {**met, 'ext_q2': ext_q2, 'alpha': best_alpha}



  Ridge Optimised — first_order_rate
  Features  : 8 (after cleaning)
  R² (train): 0.4258   (in-sample)
  Q² (LOO)  : 0.2709   ← report this
  Spearman ρ: 0.5939   (p=7e-06)
  RMSE(log) : 1.4793
  Gap R²-Q² : 0.1548   [OK]
  Permut. p : 0.0000   → SIGNIFICANT
  Best alpha: 29.9935
  Top 5 features:
    matrix_CED: +0.4632  [+0.233,+0.546]
    Hemicellulose_Percentage: +0.2064  [+0.040,+0.335]
    logP_matrix: -0.1965  [-0.335,+0.006] [CI crosses zero]
    tortuosity: -0.1830  [-0.448,+0.056] [CI crosses zero]
    Half life_matrix_avg: -0.1723  [-0.388,+0.125] [CI crosses zero]
  External Q²: -0.1258

  Ridge Optimised — hixson_rate
  Features  : 8 (after cleaning)
  R² (train): 0.4417   (in-sample)
  Q² (LOO)  : 0.2887   ← report this
  Spearman ρ: 0.598   (p=6e-06)
  RMSE(log) : 1.4262
  Gap R²-Q² : 0.153   [OK]
  Permut. p : 0.0000   → SIGNIFICANT
  Best alpha: 27.9974
  Top 5 features:
    matrix_CED: +0.4661  [+0.229,+0.555]
    Hemicellulose_Percentage: +0.2128  [+0.053,+0.336]


## 8. Final Model Comparison

In [ ]:

print(f"\n{'='*80}")
print(f"  FINAL MODEL COMPARISON")
print(f"  Dataset: {X_train.shape[0]} training + {X_test.shape[0]} external test | {len(feats_model)} features")
print(f"  Features: {feats_model}")
print(f"{'='*80}")
print(f"  {'Model':<22} {'R²(fo)':>7} {'Q²(fo)':>7} {'Gap':>6} | {'R²(hix)':>7} {'Q²(hix)':>7}")
print(f"  {'-'*65}")

pairs = [
    ('GPR Basic',        'GPR Basic fir', 'GPR Basic hix'),
    ('GPR Pearson',    'GPR Pearson fir',      'GPR Pearson hix'),
    ('SVR Basic',        'SVR Basic fir', 'SVR Basic hix'),
    ('SVR Pearson',    'SVR Pearson fir',      'SVR Pearson hix'),
    ('PLS Basic',        'PLS Basic fir', 'PLS Basic hix'),
    ('PLS Optimised',    'PLS Optimised fir',      'PLS Optimised hix'),
    ('RF Basic',         'RF Basic fir',  'RF Basic hix'),
    ('RF Pearson',       'RF Pearson fir',    'RF Pearson hix'),
    ('Lasso Basic',      'Lasso Basic fir','Lasso Basic hix'),
    ('Lasso Pearson',    'Lasso Pearson fir', 'Lasso Pearson hix'),
    ('EN Basic',         'EN Basic fir',  'EN Basic hix'),
    ('EN Optimised',       'EN opt fir',    'EN opt hix'),
    ('Ridge Basic',      'Ridge Basic fir','Ridge Basic hix'),
    ('Ridge Optimised',    'Ridge opt fir', 'Ridge opt hix'),
]
for label, kfo, khix in pairs:
    fo  = RESULTS.get(kfo, {})
    hix = RESULTS.get(khix, {})
    q2fo  = fo.get('Q2',  '--')
    star  = ' *' if isinstance(q2fo, float) and q2fo >= 0.50 else ''
    print(f"  {label:<22} {str(fo.get('R2','--')):>7} {str(q2fo):>7}{star:<2} {str(fo.get('gap','--')):>6} | "
          f"{str(hix.get('R2','--')):>7} {str(hix.get('Q2','--')):>7}")

print(f"\n  * = Q² ≥ 0.50 (useful model threshold)")
print(f"  Gap = R²−Q²: <0.20 healthy, 0.20–0.30 borderline, >0.30 overfit")
print(f"  All Q² = true LOO cross-validation | All scalers fitted inside each fold")



  FINAL MODEL COMPARISON
  Dataset: 49 training + 9 external test | 8 features
  Features: ['Half life_matrix_avg', 'Hemicellulose_Percentage', 'tortuosity', 'logP_matrix', 'interface_rot', 'fiber_hansen_sol_parameter', 'matrix_CED', 'fiber_dia_or_average_particle_size_mm']
  Model                   R²(fo)  Q²(fo)    Gap | R²(hix) Q²(hix)
  -----------------------------------------------------------------
  GPR Basic               0.9699  0.5374 * 0.4326 |  0.9708  0.5592
  GPR Optimised           0.8166  0.2632   0.5534 |   0.741  0.2887
  SVR Basic               0.7855  0.4586   0.3269 |  0.7949  0.4771
  SVR Optimised           0.7369  0.2874   0.4495 |  0.7957  0.3129
  PLS Basic               0.3705  0.2735   0.0969 |   0.384    0.29
  PLS Optimised           0.4572  0.2485   0.2087 |  0.4693  0.2388
  RF Basic                0.8128  0.4308    0.382 |  0.8171  0.4521
  RF Optimised              0.93  0.3925   0.5375 |  0.9316  0.4223
  Lasso Basic             0.4732  0.2248   0.2

Extended Metrics Table
Append after the Final Model Comparison cell
Two cells below:

Cell M1 — Comprehensive table across all models (uses existing RESULTS dict, no re-computation)
Cell M2 — Full extended metrics for GPR Basic (best model) — re-runs LOO, computes every QSAR metric required for publication

Extended Metrics Table
Append after the Final Model Comparison cell
Two cells below:

Cell M1 — Comprehensive table across all models (uses existing RESULTS dict, no re-computation)
Cell M2 — Full extended metrics for GPR Basic (best model) — re-runs LOO, computes every QSPR metric required for publication

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Build table from RESULTS dict ────────────────────────────────────────────
# All metric keys available in RESULTS (populated by each model cell)
MODEL_PAIRS = [
    ('GPR Basic',        'GPR Basic fir',   'GPR Basic hix'),
    ('GPR Pearson',    'GPR Pearson fir',      'GPR Pearson hix'),
    ('SVR Basic',        'SVR Basic fir',    'SVR Basic hix'),
    ('SVR Optimised',    'SVR Opt fir',      'SVR Opt hix'),
    ('PLS Basic',        'PLS Basic fir',    'PLS Basic hix'),
    ('PLS Optimised',    'PLS Opt fir',      'PLS Opt hix'),
    ('RF Basic',         'RF Basic fir',     'RF Basic hix'),
    ('RF Pearson',       'RF Pearson fir',    'RF Pearson hix'),
    ('Lasso Basic',      'Lasso Basic fir',  'Lasso Basic hix'),
    ('Lasso Optimised',  'Lasso Opt fir',    'Lasso Opt hix'),
    ('EN Basic',         'EN Basic fir',     'EN Basic hix'),
    ('EN Optimised',     'EN Opt fir',       'EN Opt hix'),
    ('Ridge Basic',      'Ridge Basic fir',  'Ridge Basic hix'),
    ('Ridge Optimised',  'Ridge Opt fir',    'Ridge Opt hix'),
]

def get(d, key, default='—'):
    v = d.get(key, default)
    return default if v is None else v

rows = []
for model_name, key_fo, key_hix in MODEL_PAIRS:
    fo  = RESULTS.get(key_fo,  {})
    hix = RESULTS.get(key_hix, {})
    if not fo and not hix:
        continue   # skip models that weren't run

    # RMSE real scale: exp(RMSE_log) gives the multiplicative error factor
    rmse_fo_log  = fo.get('RMSE',  None)
    rmse_hix_log = hix.get('RMSE', None)
    rmse_fo_real  = round(np.exp(rmse_fo_log),  3) if rmse_fo_log  is not None else '—'
    rmse_hix_real = round(np.exp(rmse_hix_log), 3) if rmse_hix_log is not None else '—'

    # Pearson r from Spearman ρ is NOT derivable — placeholder column
    # (computed exactly in Cell M2 for GPR)

    rows.append({
        'Model':             model_name,
        # ── first_order_rate ─────────────────────────────────────────────────
        'R²_fo':             get(fo, 'R2'),
        'Q²_fo':             get(fo, 'Q2'),
        'ρ_fo':              get(fo, 'rho'),
        'RMSE_log_fo':       get(fo, 'RMSE'),
        'RMSE_×real_fo':     rmse_fo_real,
        'Gap_fo':            get(fo, 'gap'),
        'Perm_p_fo':         get(fo, 'perm_p', '—'),
        # ── hixson_rate ──────────────────────────────────────────────────────
        'R²_hix':            get(hix, 'R2'),
        'Q²_hix':            get(hix, 'Q2'),
        'ρ_hix':             get(hix, 'rho'),
        'RMSE_log_hix':      get(hix, 'RMSE'),
        'RMSE_×real_hix':    rmse_hix_real,
        'Gap_hix':           get(hix, 'gap'),
        'Perm_p_hix':        get(hix, 'perm_p', '—'),
    })

df_metrics = pd.DataFrame(rows).set_index('Model')

# ── Column groups for readability ─────────────────────────────────────────────
cols_fo  = ['R²_fo',  'Q²_fo',  'ρ_fo',  'RMSE_log_fo',  'RMSE_×real_fo',  'Gap_fo',  'ExtQ²_fo',  'Perm_p_fo']
cols_hix = ['R²_hix', 'Q²_hix', 'ρ_hix', 'RMSE_log_hix', 'RMSE_×real_hix', 'Gap_hix', 'ExtQ²_hix', 'Perm_p_hix']

print("=" * 130)
print("  COMPREHENSIVE METRICS — first_order_rate")
print("=" * 130)
print(f"  {'Model':<22} {'R²':>7} {'Q²':>7} {'ρ':>7} {'RMSE(log)':>10} {'RMSE(×)':>9} {'Gap':>7} {'Ext Q²':>8} {'Perm p':>8}")
print(f"  {'-'*100}")
for model, row in df_metrics.iterrows():
    q2_val = row['Q²_fo']
    star   = ' ★' if isinstance(q2_val, float) and q2_val >= 0.50 else ''
    print(f"  {model:<22} {str(row['R²_fo']):>7} {str(q2_val):>7}{star:<2} "
          f"{str(row['ρ_fo']):>7} {str(row['RMSE_log_fo']):>10} {str(row['RMSE_×real_fo']):>9} "
          f"{str(row['Gap_fo']):>7} {str(row['ExtQ²_fo']):>8} {str(row['Perm_p_fo']):>8}")

print()
print("=" * 130)
print("  COMPREHENSIVE METRICS — hixson_rate")
print("=" * 130)
print(f"  {'Model':<22} {'R²':>7} {'Q²':>7} {'ρ':>7} {'RMSE(log)':>10} {'RMSE(×)':>9} {'Gap':>7} {'Ext Q²':>8} {'Perm p':>8}")
print(f"  {'-'*100}")
for model, row in df_metrics.iterrows():
    q2_val = row['Q²_hix']
    star   = ' ★' if isinstance(q2_val, float) and q2_val >= 0.50 else ''
    print(f"  {model:<22} {str(row['R²_hix']):>7} {str(q2_val):>7}{star:<2} "
          f"{str(row['ρ_hix']):>7} {str(row['RMSE_log_hix']):>10} {str(row['RMSE_×real_hix']):>9} "
          f"{str(row['Gap_hix']):>7} {str(row['ExtQ²_hix']):>8} {str(row['Perm_p_hix']):>8}")

print("""
  Column guide
  ─────────────────────────────────────────────────────────────────────────────
  R²          In-sample coefficient of determination (optimistic — always high for GPR)
  Q²          LOO predictive R² 
  ρ           Spearman rank correlation (LOO) — quality of fast-vs-slow screening
  RMSE(log)   Root Mean Squared Error in log space (LOO)
  RMSE(×)     Back-transformed RMSE: predicted real-scale rate is ×RMSE off on average
              e.g. ×3.2 means prediction is 3.2× too high or too low typically
  Gap         R² − Q² : <0.20 healthy | 0.20–0.30 borderline | >0.30 overfit
  Ext Q²      Q² on 9 Kennard-Stone external test samples (independent validation)
  Perm p      Permutation test p-value: p<0.05 confirms signal > chance
             Q² ≥ 0.50 (model to be considered for QSAR)
  ─────────────────────────────────────────────────────────────────────────────
  Note: MAE, Pearson r, CCC, bias, and Golbraikh-Tropsha k/k' are in Cell M2
  (computed for GPR Basic only — these require storing LOO predictions)
""")

# Save to CSV and Excel
df_metrics.to_csv('metrics_all_models.csv')
print("  Saved: metrics_all_models.csv")

try:
    df_metrics.to_excel('metrics_all_models.xlsx')
    print("  Saved: metrics_all_models.xlsx")
except Exception as e:
    print(f"  Excel save skipped ({e}) — CSV saved instead")


  COMPREHENSIVE METRICS — first_order_rate
  Model                       R²      Q²       ρ  RMSE(log)   RMSE(×)     Gap   Ext Q²   Perm p
  ----------------------------------------------------------------------------------------------------
  GPR Basic               0.9699  0.5374 ★  0.7309     1.1784     3.249  0.4326 0.33479022334500086        —
  GPR Optimised           0.8166  0.2632    0.5361     1.4871     4.424  0.5534        —        —
  SVR Basic               0.7855  0.4586    0.6669     1.2748     3.578  0.3269 0.46703474560578206        —
  SVR Optimised           0.7369  0.2874    0.4902     1.4625     4.317  0.4495        —        —
  PLS Basic               0.3705  0.2735    0.4872     1.4767     4.378  0.0969 -0.5999579417518313        —
  PLS Optimised           0.4572  0.2485    0.5125     1.5018      4.49  0.2087        —        —
  RF Basic                0.8128  0.4308    0.6604      1.307     3.695   0.382 0.2716887533123531        —
  RF Optimised              0